# Day 072 — Solution: Audio Transcriber

In [ ]:
_TRANSCRIBER_SRC = '"""audio_transcriber.py — Day 072: Speech-to-Text.\n\nTranscribes audio files using the openai-whisper local package.\nopenai-whisper is the LOCAL package — it runs entirely on your machine.\nNo API key, no network calls after the model is downloaded.\n\nSetup:\n    pip install openai-whisper\n    brew install ffmpeg      # macOS\n    # Ubuntu: apt install ffmpeg\n\nUsage:\n    from audio_transcriber import AudioTranscriber\n\n    # Real transcription\n    tr = AudioTranscriber(model=\'base\')\n    result = tr.transcribe(\'recording.mp3\')\n    print(result[\'text\'])\n\n    # Testing — no audio file needed\n    mock = lambda src: {\'text\': \' Hello.\', \'language\': \'en\', \'segments\': [\n        {\'id\': 0, \'start\': 0.0, \'end\': 1.0, \'text\': \' Hello.\',\n         \'avg_logprob\': -0.2, \'no_speech_prob\': 0.01}]}\n    tr = AudioTranscriber(transcribe_fn=mock)\n    print(tr.get_text(b\'fake audio\'))   # Hello.\n"""\nimport os\nimport tempfile\nfrom typing import Callable, Optional, Union\n\n\ndef _format_time(seconds: float) -> str:\n    """Convert seconds to HH:MM:SS string."""\n    m, s = divmod(int(seconds), 60)\n    h, m = divmod(m, 60)\n    return f"{h:02d}:{m:02d}:{s:02d}"\n\n\ndef format_transcript(result: dict, include_timestamps: bool = False) -> str:\n    """Format a whisper result dict as a readable string.\n\n    Args:\n        result:             Whisper result dict (keys: text, segments, language)\n        include_timestamps: If True, prefix each segment with [HH:MM:SS]\n    Returns:\n        Formatted transcript string\n    """\n    if not include_timestamps:\n        return result.get("text", "").strip()\n    lines = []\n    for seg in result.get("segments", []):\n        ts = _format_time(seg.get("start", 0.0))\n        lines.append(f"[{ts}] {seg.get(\'text\', \'\').strip()}")\n    return "\\n".join(lines)\n\n\ndef extract_segments(result: dict) -> list:\n    """Extract time-stamped segments from a whisper result.\n\n    Returns:\n        list of dicts: {start: float, end: float, text: str, confidence: float}\n        confidence is derived from avg_logprob ∈ (-∞, 0]: 0.0 = poor, 1.0 = perfect\n    """\n    out = []\n    for seg in result.get("segments", []):\n        logprob = seg.get("avg_logprob", -1.0)\n        confidence = min(1.0, max(0.0, 1.0 + logprob))\n        out.append({\n            "start":      float(seg.get("start", 0.0)),\n            "end":        float(seg.get("end", 0.0)),\n            "text":       seg.get("text", "").strip(),\n            "confidence": round(confidence, 4),\n        })\n    return out\n\n\ndef transcribe_audio(source, transcribe_fn: Optional[Callable] = None,\n                     model: str = "base") -> dict:\n    """Transcribe an audio source using openai-whisper.\n\n    Args:\n        source:        File path (str/Path), bytes, or numpy float32 array\n        transcribe_fn: callable(source) -> dict for testing (no whisper needed)\n        model:         Whisper model size: tiny, base, small, medium, large\n    Returns:\n        dict with keys: text (str), language (str), segments (list)\n    """\n    if transcribe_fn is not None:\n        return transcribe_fn(source)\n    import whisper as _whisper\n    mdl = _whisper.load_model(model)\n    if isinstance(source, (bytes, bytearray)):\n        with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as f:\n            f.write(source)\n            tmp = f.name\n        try:\n            return mdl.transcribe(tmp)\n        finally:\n            os.unlink(tmp)\n    return mdl.transcribe(str(source))\n\n\ndef search_transcript(result: dict, query: str,\n                      case_sensitive: bool = False) -> list:\n    """Find segments whose text contains the query string.\n\n    Args:\n        result:         Whisper result dict\n        query:          Search string\n        case_sensitive: If False, comparison is case-insensitive\n    Returns:\n        list of matching segment dicts (same format as extract_segments)\n    """\n    segments = extract_segments(result)\n    if not case_sensitive:\n        q = query.lower()\n        return [s for s in segments if q in s["text"].lower()]\n    return [s for s in segments if query in s["text"]]\n\n\nclass AudioTranscriber:\n    """Transcribe audio files using openai-whisper.\n\n    Inject transcribe_fn for testing without whisper or audio files::\n\n        mock = lambda src: {\'text\': \' Hello.\', \'language\': \'en\',\n                            \'segments\': [{\'id\':0,\'start\':0.0,\'end\':1.0,\n                             \'text\':\' Hello.\',\'avg_logprob\':-0.2,\'no_speech_prob\':0.01}]}\n        tr = AudioTranscriber(transcribe_fn=mock)\n    """\n\n    def __init__(self, model: str = "base",\n                 transcribe_fn: Optional[Callable] = None) -> None:\n        self._model        = model\n        self._transcribe_fn = transcribe_fn\n\n    def transcribe(self, source) -> dict:\n        """Transcribe audio. Returns full whisper result dict."""\n        return transcribe_audio(source,\n                                transcribe_fn=self._transcribe_fn,\n                                model=self._model)\n\n    def get_text(self, source) -> str:\n        """Return plain transcript text (stripped)."""\n        return format_transcript(self.transcribe(source))\n\n    def get_segments(self, source) -> list:\n        """Return list of time-stamped segment dicts."""\n        return extract_segments(self.transcribe(source))\n\n    def search(self, source, query: str,\n               case_sensitive: bool = False) -> list:\n        """Search for a query in the transcript segments."""\n        return search_transcript(self.transcribe(source), query,\n                                 case_sensitive=case_sensitive)\n'
from pathlib import Path
Path('audio_transcriber.py').write_text(_TRANSCRIBER_SRC, encoding='utf-8')
print('audio_transcriber.py written.')

In [ ]:
from audio_transcriber import (
    _format_time, format_transcript, extract_segments,
    transcribe_audio, search_transcript, AudioTranscriber,
)

_MOCK = {
    'text': ' Hello world. Testing one two three.',
    'language': 'en',
    'segments': [
        {'id': 0, 'start': 0.0, 'end': 2.5, 'text': ' Hello world.',
         'avg_logprob': -0.25, 'no_speech_prob': 0.01},
        {'id': 1, 'start': 2.5, 'end': 6.0, 'text': ' Testing one two three.',
         'avg_logprob': -0.20, 'no_speech_prob': 0.01},
    ],
}
_mock_fn = lambda src: _MOCK

# 1. _format_time
assert _format_time(0.0)   == '00:00:00'
assert _format_time(65.0)  == '00:01:05'
assert _format_time(3661.0)== '01:01:01'
print("\u2705 _format_time correct")

# 2. format_transcript
t = format_transcript(_MOCK)
assert t == 'Hello world. Testing one two three.'
ts = format_transcript(_MOCK, include_timestamps=True)
assert '[00:00:00]' in ts and '[00:00:02]' in ts
print("\u2705 format_transcript correct")

# 3. extract_segments
segs = extract_segments(_MOCK)
assert len(segs) == 2
assert segs[0] == {'start': 0.0, 'end': 2.5, 'text': 'Hello world.', 'confidence': 0.75}
print("\u2705 extract_segments correct")

# 4. transcribe_audio
result = transcribe_audio(b'fake', transcribe_fn=_mock_fn)
assert result['language'] == 'en' and len(result['text']) > 0
print("\u2705 transcribe_audio correct")

# 5. search_transcript
hits = search_transcript(_MOCK, 'hello')
assert len(hits) == 1 and hits[0]['start'] == 0.0
no_hits = search_transcript(_MOCK, 'HELLO', case_sensitive=True)
assert no_hits == []
print("\u2705 search_transcript correct")

# 6. AudioTranscriber
tr = AudioTranscriber(transcribe_fn=_mock_fn)
assert tr.get_text(b'a') == 'Hello world. Testing one two three.'
s = tr.get_segments(b'a')
assert len(s) == 2 and s[0]['confidence'] == 0.75
h = tr.search(b'a', 'testing')
assert len(h) == 1 and abs(h[0]['start'] - 2.5) < 0.001
print("\u2705 AudioTranscriber correct")

print("\nAudio Transcriber complete!")
